In [ ]:
# Paths configuration for PSG dataset
dataset_path = "datasets/psg/YOLO_anno"
yaml_path = "datasets/psg/YOLO_anno/data.yaml"

# Model trained on PSG (sẽ được tạo sau khi train YOLO11)
model_path = "checkpoints/PSG/yolo11m_psg/weights/best.pt"

import os
from ultralytics import YOLO

# Load trained model
if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"Model loaded: {model_path}")
else:
    print(f"⚠️ Model not found: {model_path}")
    print("Train YOLO11 first using my_script/psg_to_yolo.ipynb")

from ultralytics.cfg import get_cfg

cfg = get_cfg()

# Load train data
from ultralytics.utils import yaml_load
from ultralytics.data import build_dataloader, build_yolo_dataset

if os.path.exists(yaml_path):
    data = yaml_load(yaml_path)
    
    # Load dataset
    splits = [data['train'], data['val'], data.get('test', data['val'])]
    
    split = splits[0]  # Use train split
    img_dir = os.path.join(split, 'images')
    dataset = build_yolo_dataset(cfg, img_dir, 1, data)
    
    print(f"Dataset loaded: {len(dataset)} images")
else:
    print(f"⚠️ YAML not found: {yaml_path}")
    print("Convert dataset first using my_script/psg_to_yolo.ipynb")

/home/maelic/miniconda3/envs/phd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# YOLO Data Augmentation

**Purpose**: Tự động thêm objects vào annotations bằng cách sử dụng trained YOLO model

**How it works**:
1. Load YOLO model đã train trên PSG
2. Chạy model trên toàn bộ train set
3. Lọc predictions: chỉ giữ boxes **không overlap** với ground truth (IoU < threshold)
4. Thêm boxes mới vào annotation files

**Result**: Dataset với nhiều labeled objects hơn → Train tốt hơn

**Prerequisites**:
- ✅ Dataset đã convert sang YOLO format
- ✅ YOLO model đã train xong
- ✅ Model accuracy tốt (>0.7 mAP)

# Helper Functions

In [ ]:
import torch

def bbox_iou(box1, box2, xywh=True, eps=1e-7):
    """
    Calculate IoU between boxes
    Args:
        box1: predictions (xyxy or xywh format)
        box2: ground truth boxes (xywh format from YOLO)
        xywh: if True, box2 is in xywh format
    Returns:
        iou values for each box1 against all box2
    """
    if xywh:
        # Convert xywh to xyxy
        # box2 format: [x_center, y_center, w, h]
        box2_xyxy = torch.zeros_like(box2)
        box2_xyxy[:, 0] = box2[:, 0] - box2[:, 2] / 2  # x1
        box2_xyxy[:, 1] = box2[:, 1] - box2[:, 3] / 2  # y1
        box2_xyxy[:, 2] = box2[:, 0] + box2[:, 2] / 2  # x2
        box2_xyxy[:, 3] = box2[:, 1] + box2[:, 3] / 2  # y2
        box2 = box2_xyxy
    
    # Get coordinates
    b1_x1, b1_y1, b1_x2, b1_y2 = box1[0], box1[1], box1[2], box1[3]
    b2_x1, b2_y1, b2_x2, b2_y2 = box2[:, 0], box2[:, 1], box2[:, 2], box2[:, 3]
    
    # Intersection area
    inter_x1 = torch.max(b1_x1, b2_x1)
    inter_y1 = torch.max(b1_y1, b2_y1)
    inter_x2 = torch.min(b1_x2, b2_x2)
    inter_y2 = torch.min(b1_y2, b2_y2)
    
    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
    
    # Union area
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area + eps
    
    # IoU
    iou = inter_area / union_area
    
    return iou

print("bbox_iou function defined")

In [ ]:
# Create output directory for augmented dataset
new_dataset_path = dataset_path.strip().replace('YOLO_anno', 'YOLO_anno_augmented')
os.makedirs(new_dataset_path, exist_ok=True)

print(f"Output directory: {new_dataset_path}")

# Run Augmentation

**Parameters**:
- `iou_thres`: Nếu prediction có IoU < 0.5 với TẤT CẢ ground truth boxes → object mới
- `conf_thres`: Chỉ giữ predictions có confidence > 0.5

In [ ]:
from tqdm import tqdm

# Thresholds
iou_thres = 0.5      # Objects mới phải có IoU < 0.5 với tất cả GT boxes
conf_thres = 0.5     # Chỉ giữ predictions có confidence > 0.5

dest_anno_path = new_dataset_path + '/train/labels'
os.makedirs(dest_anno_path, exist_ok=True)

# Copy images
dest_img_path = new_dataset_path + '/train/images'
os.makedirs(dest_img_path, exist_ok=True)

num_total = 0
num_images_with_new_objects = 0

# Run model on dataset
for i, img in enumerate(tqdm(dataset, desc="Augmenting annotations")):
    # Get image name
    img_name = img['im_file'].split('/')[-1].split('.')[0]
    
    # Get annotation file
    anno_file = dataset.label_files[i]
    
    # Read annotation file
    with open(anno_file, 'r') as f:
        anno = f.readlines()
    
    original_count = len(anno)
    
    # Get ground truth boxes
    gt_boxes = img['bboxes'].cuda() if torch.cuda.is_available() else img['bboxes']
    
    # Forward pass
    results = model(img['im_file'], verbose=False)
    
    # Check IoU on GT boxes and predictions
    new_objects_this_image = 0
    for j, box in enumerate(results[0].boxes):
        # Calculate IoU with all GT boxes
        iou = bbox_iou(box.xyxy[0], gt_boxes, xywh=True)
        
        # If this box doesn't overlap with any GT box
        if len(gt_boxes) == 0 or all(iou < iou_thres):
            # And has high confidence
            if box.conf > conf_thres:
                # Add to annotation
                bbox = box.xyxyn[0]  # Normalized xyxy format
                # Convert to xywh normalized
                x_center = (bbox[0] + bbox[2]) / 2
                y_center = (bbox[1] + bbox[3]) / 2
                width = bbox[2] - bbox[0]
                height = bbox[3] - bbox[1]
                
                anno.append(f"{int(box.cls.item())} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                num_total += 1
                new_objects_this_image += 1
    
    if new_objects_this_image > 0:
        num_images_with_new_objects += 1
    
    # Write out to destination
    with open(dest_anno_path + f'/{img_name}.txt', 'w') as f:
        f.writelines(anno)
    
    # Copy image
    import shutil
    shutil.copy(img['im_file'], dest_img_path + f'/{img_name}.jpg')

print(f"\n✅ Augmentation complete!")
print(f"📊 Statistics:")
print(f"  - Total new objects added: {num_total}")
print(f"  - Images with new objects: {num_images_with_new_objects}/{len(dataset)}")
print(f"  - Average new objects per image: {num_total/len(dataset):.2f}")
print(f"  - Output: {dest_anno_path}")

100%|██████████| 11739/11739 [08:18<00:00, 23.56it/s]

Total new objects added: 41194


# Verify Results

In [ ]:
# Display overall statistics
num_objects = 0
num_files = 0

for anno_file in os.listdir(dest_anno_path):
    if anno_file.endswith('.txt'):
        with open(os.path.join(dest_anno_path, anno_file), 'r') as f:
            anno = f.readlines()
            num_objects += len(anno)
            num_files += 1

print(f"📈 Final Dataset Statistics:")
print(f"  - Total annotation files: {num_files}")
print(f"  - Total objects (original + augmented): {num_objects}")
print(f"  - Average objects per image: {num_objects/num_files:.2f}")
print(f"\n💡 Compare with original dataset to see the increase!")

# Visualize Example

Xem một ví dụ để verify objects mới được thêm đúng

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

# Pick a random image
anno_files = [f for f in os.listdir(dest_anno_path) if f.endswith('.txt')]
sample_file = random.choice(anno_files)
img_name = sample_file.replace('.txt', '.jpg')

# Load image
img_path = os.path.join(dest_img_path, img_name)
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

# Load annotations
anno_path = os.path.join(dest_anno_path, sample_file)
with open(anno_path, 'r') as f:
    boxes = f.readlines()

# Load original annotations for comparison
orig_anno_path = os.path.join(dataset_path, 'train/labels', sample_file)
with open(orig_anno_path, 'r') as f:
    orig_boxes = f.readlines()

print(f"Image: {img_name}")
print(f"Original boxes: {len(orig_boxes)}")
print(f"Augmented boxes: {len(boxes)}")
print(f"New boxes added: {len(boxes) - len(orig_boxes)}")

# Draw boxes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Original
img_orig = img.copy()
for box in orig_boxes:
    parts = box.strip().split()
    cls_id = int(parts[0])
    x_c, y_c, bw, bh = map(float, parts[1:5])
    
    x1 = int((x_c - bw/2) * w)
    y1 = int((y_c - bh/2) * h)
    x2 = int((x_c + bw/2) * w)
    y2 = int((y_c + bh/2) * h)
    
    cv2.rectangle(img_orig, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img_orig, str(cls_id), (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Augmented
img_aug = img.copy()
for i, box in enumerate(boxes):
    parts = box.strip().split()
    cls_id = int(parts[0])
    x_c, y_c, bw, bh = map(float, parts[1:5])
    
    x1 = int((x_c - bw/2) * w)
    y1 = int((y_c - bh/2) * h)
    x2 = int((x_c + bw/2) * w)
    y2 = int((y_c + bh/2) * h)
    
    # Original boxes in green, new boxes in red
    color = (0, 255, 0) if i < len(orig_boxes) else (255, 0, 0)
    thickness = 2 if i < len(orig_boxes) else 3
    
    cv2.rectangle(img_aug, (x1, y1), (x2, y2), color, thickness)
    cv2.putText(img_aug, str(cls_id), (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

ax1.imshow(img_orig)
ax1.set_title(f'Original ({len(orig_boxes)} objects)', fontsize=14)
ax1.axis('off')

ax2.imshow(img_aug)
ax2.set_title(f'Augmented ({len(boxes)} objects)\nGreen=Original, Red=New', fontsize=14)
ax2.axis('off')

plt.tight_layout()
plt.show()